# Workforce Data Validation

This notebook runs Story 02 schema validation and basic quality profiling for the four raw workforce CSV files.

In [1]:
from pathlib import Path
import sys

import polars as pl
import yaml
from IPython.display import Markdown, display

workspace_root = next(path for path in [Path.cwd(), *Path.cwd().parents] if (path / 'shared').exists() and (path / 'artifacts').exists())
if str(workspace_root) not in sys.path:
    sys.path.insert(0, str(workspace_root))

from shared.src.data_processing.workforce_validation import validate_workforce_files

raw_dir = workspace_root / 'shared' / 'data' / '1_raw' / 'workforce'
file_paths = [
    raw_dir / 'doctors.csv',
    raw_dir / 'nurses.csv',
    raw_dir / 'pharmacists.csv',
    raw_dir / 'physiotherapists.csv',
]
file_paths

[PosixPath('/Users/alfredtang/Documents/Projects/gen-e2/gen-e2-data-analysis/shared/data/1_raw/workforce/doctors.csv'),
 PosixPath('/Users/alfredtang/Documents/Projects/gen-e2/gen-e2-data-analysis/shared/data/1_raw/workforce/nurses.csv'),
 PosixPath('/Users/alfredtang/Documents/Projects/gen-e2/gen-e2-data-analysis/shared/data/1_raw/workforce/pharmacists.csv'),
 PosixPath('/Users/alfredtang/Documents/Projects/gen-e2/gen-e2-data-analysis/shared/data/1_raw/workforce/physiotherapists.csv')]

In [2]:
results = validate_workforce_files(file_paths)
results

2026-04-22 23:16:08.566 | INFO     | shared.src.data_processing.workforce_validation:validate_workforce_file:55 - Validating workforce file: /Users/alfredtang/Documents/Projects/gen-e2/gen-e2-data-analysis/shared/data/1_raw/workforce/doctors.csv
2026-04-22 23:16:08.570 | INFO     | shared.src.data_processing.workforce_validation:validate_workforce_file:126 - doctors.csv schema=pass duplicates=0 negative_count_rows=0 year_range=2006..2019
2026-04-22 23:16:08.571 | INFO     | shared.src.data_processing.workforce_validation:validate_workforce_file:55 - Validating workforce file: /Users/alfredtang/Documents/Projects/gen-e2/gen-e2-data-analysis/shared/data/1_raw/workforce/nurses.csv
2026-04-22 23:16:08.573 | INFO     | shared.src.data_processing.workforce_validation:validate_workforce_file:126 - nurses.csv schema=pass duplicates=0 negative_count_rows=0 year_range=2006..2019
2026-04-22 23:16:08.573 | INFO     | shared.src.data_processing.workforce_validation:validate_workforce_file:55 - Vali

{'doctors': {'file_name': 'doctors.csv',
  'file_path': '/Users/alfredtang/Documents/Projects/gen-e2/gen-e2-data-analysis/shared/data/1_raw/workforce/doctors.csv',
  'row_count': 78,
  'column_count': 4,
  'source_columns': ['profession', 'sector', 'year', 'headcount'],
  'matched_columns': {'year': 'year',
   'sector': 'sector',
   'count': 'headcount'},
  'missing_required_columns': [],
  'schema_validation': 'pass',
  'null_counts': {'year': 0, 'sector': 0, 'count': 0},
  'null_rates': {'year': 0.0, 'sector': 0.0, 'count': 0.0},
  'duplicate_row_count': 0,
  'negative_count_rows': 0,
  'unique_year_values': [2006,
   2007,
   2008,
   2009,
   2010,
   2011,
   2012,
   2013,
   2014,
   2015,
   2016,
   2017,
   2018,
   2019],
  'unique_sector_values': ['not in active practice', 'private', 'public'],
  'year_range': {'min': 2006, 'max': 2019}},
 'nurses': {'file_name': 'nurses.csv',
  'file_path': '/Users/alfredtang/Documents/Projects/gen-e2/gen-e2-data-analysis/shared/data/1_raw

In [3]:
summary_rows = []
for profession, finding in results.items():
    summary_rows.append({
        'file': finding['file_name'],
        'schema_validation': finding['schema_validation'],
        'rows': finding['row_count'],
        'duplicates': finding.get('duplicate_row_count', 0),
        'negative_count_rows': finding.get('negative_count_rows', 0),
        'year_min': finding.get('year_range', {}).get('min'),
        'year_max': finding.get('year_range', {}).get('max'),
        'sectors': ', '.join(finding.get('unique_sector_values', [])),
    })

summary_df = pl.DataFrame(summary_rows)
summary_df

file,schema_validation,rows,duplicates,negative_count_rows,year_min,year_max,sectors
str,str,i64,i64,i64,i64,i64,str
"""doctors.csv""","""pass""",78,0,0,2006,2019,"""not in active practice, privat…"
"""nurses.csv""","""pass""",126,0,0,2006,2019,"""not in active practice, privat…"
"""pharmacists.csv""","""pass""",42,0,0,2006,2019,"""not in active practice, privat…"
"""physiotherapists.csv""","""pass""",18,0,0,2014,2019,"""not in active practice, privat…"


In [4]:
display(Markdown('## YAML Preview'))
yaml.safe_dump(results, sort_keys=False)

## YAML Preview

'doctors:\n  file_name: doctors.csv\n  file_path: /Users/alfredtang/Documents/Projects/gen-e2/gen-e2-data-analysis/shared/data/1_raw/workforce/doctors.csv\n  row_count: 78\n  column_count: 4\n  source_columns:\n  - profession\n  - sector\n  - year\n  - headcount\n  matched_columns:\n    year: year\n    sector: sector\n    count: headcount\n  missing_required_columns: []\n  schema_validation: pass\n  null_counts:\n    year: 0\n    sector: 0\n    count: 0\n  null_rates:\n    year: 0.0\n    sector: 0.0\n    count: 0.0\n  duplicate_row_count: 0\n  negative_count_rows: 0\n  unique_year_values:\n  - 2006\n  - 2007\n  - 2008\n  - 2009\n  - 2010\n  - 2011\n  - 2012\n  - 2013\n  - 2014\n  - 2015\n  - 2016\n  - 2017\n  - 2018\n  - 2019\n  unique_sector_values:\n  - not in active practice\n  - private\n  - public\n  year_range:\n    min: 2006\n    max: 2019\nnurses:\n  file_name: nurses.csv\n  file_path: /Users/alfredtang/Documents/Projects/gen-e2/gen-e2-data-analysis/shared/data/1_raw/workforce/